In [1]:
!pip install kaggle


   ---------- ----------------------------- 1/4 [python-slugify]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   ------------------------------ --------- 3/4 [kaggle]
   ------------------------------ --------- 3/4 [kaggle]
   ------------------------------ --------- 3/4 [kaggle]
   ---------------------------------------- 4/4 [kaggle]




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os

# Kaggle -> Account -> API
# Scroll down to Legacy API Key
# Click "Create New API Key"
# This will download kaggle.json file automatically
# File will contain username and key

# Your Kaggle credentials
kaggle_json = {
    "username": "loisrandolph",
    "key": "157a0e53a6513cf87a7dc717fa7edfcd"
}

# Ensure the .kaggle folder exists
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Write kaggle.json
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_json, f)

print("kaggle.json created successfully")

kaggle.json created successfully


In [4]:
import os

# Create data/raw inside the current folder
os.makedirs("data/raw", exist_ok=True)
print("data/raw folder is ready")

data/raw folder is ready


In [5]:
!kaggle datasets download -d tobiasbueck/multilingual-customer-support-tickets -p data/raw --unzip

Dataset URL: https://www.kaggle.com/datasets/tobiasbueck/multilingual-customer-support-tickets
License(s): Attribution 4.0 International (CC BY 4.0)




  0%|          | 0.00/16.1M [00:00<?, ?B/s]
  6%|6         | 1.00M/16.1M [00:00<00:02, 6.14MB/s]
 31%|###1      | 5.00M/16.1M [00:00<00:00, 20.9MB/s]
 56%|#####5    | 9.00M/16.1M [00:00<00:00, 24.8MB/s]
 81%|########  | 13.0M/16.1M [00:00<00:00, 27.5MB/s]
 99%|#########9| 16.0M/16.1M [00:00<00:00, 25.9MB/s]
100%|##########| 16.1M/16.1M [00:00<00:00, 24.1MB/s]


In [47]:
import pandas as pd

csv_path = "data/raw/dataset-tickets-multi-lang-4-20k.csv"

df = pd.read_csv(csv_path)

In [48]:
# Inspecting data first few rows
df.head()

,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Pla...,Die Datenanalyse-Plattform brach unerwartet ab...,Ich werde Ihnen bei der Lösung des Problems he...,Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN
3,Krankenhaus-Dienstleistung-Problem,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,Zurück zur E-Mail-Beschwerde über den Sperrver...,Incident,Customer Service,high,de,Security,Breach,Login,Maintenance,Incident,Resolution,Feedback,NaN
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,Security,Customer,Compliance,Breach,Documentation,Guidance,NaN,NaN


In [49]:
# Checking for missing values/NA

missing_counts = df.isna().sum()
print("Missing values per column:\n", missing_counts)


# Keeping only columns with missing values 
missing_counts = df.isna().sum() 
missing_cols = missing_counts[missing_counts > 0].index.tolist()
print("Columns with missing values:\n", missing_cols)

Missing values per column:
 subject      1461
body            2
answer          4
type            0
queue           0
priority        0
language        0
tag_1           0
tag_2          46
tag_3          95
tag_4        1539
tag_5        6909
tag_6       12649
tag_7       16072
tag_8       18093
dtype: int64
Columns with missing values:
 ['subject', 'body', 'answer', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [50]:
# For each column with missing values
for col in missing_cols:
    # Replace NaN with "missing"
    df[col] = df[col].apply(lambda x: x if pd.notna(x) else "missing")
    
    # Get unique values including "missing"
    unique_vals = df[col].unique().tolist()
    
    # Count missing entries
    missing_count = (df[col] == "missing").sum()
    total_count = len(df)
    
    # Print summary
    print(f"\nColumn: {col}")
    print(f"Unique values (including 'missing'): {unique_vals[:10]}...")  # show first 10 only
    print(f"Missing entries: {missing_count} ({missing_count/total_count:.2%})")


Column: subject
Unique values (including 'missing'): ['Unvorhergesehener Absturz der Datenanalyse-Plattform', 'Customer Support Inquiry', 'Data Analytics for Investment', 'Krankenhaus-Dienstleistung-Problem', 'Security', 'Concerns About Securing Medical Data on 2-in-1 Convertible Laptop with Norton 360', 'Ratung für Sicherung medizinischer Daten in HubSpot CRM PostgreSQL-Umgebungen', 'Problem with Integration', 'Assistance Request', 'Support Request']...
Missing entries: 1461 (7.31%)

Column: body
Unique values (including 'missing'): ['Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe versucht, Laravel 8 und meinen MacBook Pro neu zu starten, aber das Problem behält sich bei. Ich benötige Ihre Unterstützung, um diesen Fehler zu beheben.', 'Seeking information on digital strategies that can aid in brand growth and details on the available services. Looking forward to learning more to help our business grow. Thank you, and I look forward t

In [53]:
# Threshold for categorical-like columns (adjust as needed)
max_unique_for_cat = 10

# Prepare summary list
summary = []

for col in missing_cols:
    
    # Get all unique values, including 'missing'
    unique_vals = df[col].unique().tolist()
    n_unique = len(unique_vals)
    
    # Determine if categorical-like
    is_categorical = n_unique <= max_unique_for_cat
    
    # Count missing entries (now represented as 'missing')
    missing_count = (df[col] == "missing").sum()
    total_count = len(df)
    
    # Append to summary
    summary.append({
        'Column': col,
        'Num Unique (including missing)': n_unique,
        '% Missing': missing_count/total_count,
        'Categorical-Like': is_categorical,
        'Unique Values': unique_vals
    })

# Convert summary to DataFrame for easy viewing
summary_df = pd.DataFrame(summary)
summary_df

,Column,Num Unique (including missing),% Missing,Categorical-Like,Unique Values
0,subject,18540,0.07305,False,[Unvorhergesehener Absturz der Datenanalyse-Pl...
1,body,19999,0.00010,False,[Die Datenanalyse-Plattform brach unerwartet a...
2,answer,19997,0.00020,False,[Ich werde Ihnen bei der Lösung des Problems h...
3,tag_2,205,0.00230,False,"[Technical, Sales, Product, Breach, Customer, ..."
4,tag_3,345,0.00475,False,"[Bug, IT, Guidance, Login, Compliance, Feature..."
5,tag_4,482,0.07695,False,"[Hardware, Tech Support, Documentation, Mainte..."
6,tag_5,579,0.34545,False,"[Resolution, missing, Performance, Incident, D..."
7,tag_6,567,0.63245,False,"[Outage, missing, Feature, Resolution, Guidanc..."
8,tag_7,493,0.80360,False,"[Documentation, missing, Feedback, Update, Vir..."
9,tag_8,387,0.90465,False,"[missing, Reference Number, Encryption, Incide..."


In [56]:
main_folder = "Automated-IT-Support-Ticket-Intelligence"
os.makedirs(main_folder, exist_ok=True)

import shutil
# Moving 'data' folder inside main_folder
shutil.move("data", os.path.join(main_folder, "data"))

'Automated-IT-Support-Ticket-Intelligence\\data'

In [57]:
# Creating src folder inside main project folder
os.makedirs(os.path.join(main_folder, "src"), exist_ok=True)

In [ ]:
# Moving files inside main project folder
current_path = r"C:\01_data_integration.ipynb"
target_folder = r"C:\Automated-IT-Support-Ticket-Intelligence\src"
target_path = os.path.join(target_folder, "01_data_integration.ipynb")

shutil.move(current_path, target_path)
print(f"File moved to: {target_path}")

In [65]:
os.getcwd()

'C:\\Users\\loisr'